## AutoShop inkl. MCP Server

In diesem Notebook wird das AutoShop-Webshop-Beispiel in einem Kubernetes-Cluster gestartet und um einen MCP Server erweitert. 

Zuerst werden die benötigten Microservices für Katalog, Kunden, Bestellungen und Webshop im Namespace ms-mcp deployt. 

Zusätzlich wird ein MCP Server gestartet, der als Wrapper um die bestehenden REST APIs des AutoShop-Systems implementiert ist. 

Anschliessend wird der MCP Server über Streamable HTTP getestet, indem eine ClientSession aufgebaut, die verfügbaren Tools abgefragt und ein erstes Tool exemplarisch ausgeführt wird.


In [ ]:
%%bash
kubectl create namespace ms-mcp

In [ ]:
%%bash
echo "https://"$(cat ~/data/server-ip)":30443"

In [ ]:
%%bash
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/catalog-deployment.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/customer-deployment.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/order-deployment.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/webshop-deployment.yaml 
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/catalog-service.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/customer-service.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/order-service.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/webshop-service.yaml
kubectl get pod,services --namespace ms-mcp  

Da wir keinen LoadBalancer haben müssen wir mit einem kleinen Shellscript selber die IP des Clusters und der gemappte Port (port-based-routing) als URL aufbereiten.

In [ ]:
! echo "http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace ms-mcp webshop -o=jsonpath='{ .spec.ports[0].nodePort }')/webshop

Dazu der MCP Server welcher als Wrapper implementiert ist

In [ ]:
%%bash
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/mcp-server-deployment.yaml 
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/mcp-server-service.yaml

### Testen

In [ ]:
import subprocess
import json
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


def get_server_url():
    server_ip = subprocess.check_output(
        "cat ~/data/server-ip",
        shell=True,
        text=True
    ).strip()

    node_port = subprocess.check_output(
        "kubectl get service --namespace ms-mcp autoshop-mcp-server "
        "-o=jsonpath='{ .spec.ports[0].nodePort }'",
        shell=True,
        text=True
    ).strip()

    return f"http://{server_ip}:{node_port}/mcp"


async def main():
    server_url = get_server_url()

    print(f"Verbinde mit MCP Streamable-HTTP-Server unter: {server_url}")

    try:
        async with streamablehttp_client(server_url) as (read_stream, write_stream, _):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("MCP-Session erfolgreich initialisiert")

                response = await session.list_tools()

                if not response.tools:
                    print("Server antwortet, aber es wurden keine Tools registriert.")
                    return

                print(f"Gefundene Tools ({len(response.tools)}):")
                for tool in response.tools:
                    print(f"- {tool.name}")
                    print(f"  inputSchema: {json.dumps(tool.inputSchema, ensure_ascii=False)}")

                test_tool = None
                for tool in response.tools:
                    required = tool.inputSchema.get("required", [])
                    if not required:
                        test_tool = tool
                        break

                if not test_tool:
                    print("Kein Tool ohne Pflichtparameter gefunden.")
                    print("Rufe ein Tool gezielt mit passenden Argumenten auf.")
                    return

                print(f"\nTeste Tool: {test_tool.name}")
                result = await session.call_tool(test_tool.name, arguments={})

                print("\nAntwort:")
                for item in result.content:
                    print(item)

    except Exception as e:
        print(f"Verbindungs- oder MCP-Fehler: {e}")

In [ ]:
await main()

---

### Verwenden mit AI

In [ ]:
%run ~/data/env.py
! cat ~/data/env.py

Verbindung zum OpenAI API 

In [ ]:
%run ~/data/env.py

import json
import traceback
from openai import OpenAI

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)

MCP_URL = get_server_url()

Dann eine kleine Konvertierung von MCP-Tools zu OpenAI-Tools:

In [ ]:
def mcp_tool_to_openai_tool(tool):
    schema = tool.inputSchema or {
        "type": "object",
        "properties": {},
    }

    return {
        "type": "function",
        "name": tool.name,
        "description": getattr(tool, "description", None) or f"MCP Tool {tool.name}",
        "parameters": schema,
    }


def mcp_result_to_text(result):
    parts = []

    for block in result.content:
        if hasattr(block, "text"):
            parts.append(block.text)
        elif hasattr(block, "model_dump"):
            parts.append(json.dumps(block.model_dump(), ensure_ascii=False))
        else:
            parts.append(str(block))

    return "\n".join(parts)


Read-only Toolset. order_delete ist hier bewusst nicht freigegeben.

In [ ]:
READ_ONLY_TOOLS = {
    "catalog_list_items",
    "catalog_get_item",
    "customer_list_items",
    "customer_get_item",
    "order_list_items",
    "order_get_item",
}

Der effektive Aufruf

In [ ]:
async def ask_mcp(question: str, allowed_tools=READ_ONLY_TOOLS, max_rounds: int = 8):
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()

            mcp_tools = [
                tool for tool in tools_result.tools
                if tool.name in allowed_tools
            ]

            openai_tools = [
                mcp_tool_to_openai_tool(tool)
                for tool in mcp_tools
            ]

            input_items = [
                {
                    "role": "system",
                    "content": (
                        "Du bist ein Assistent für einen Demo-Shop. "
                        "Nutze die bereitgestellten Tools für Catalog, Customer und Order. "
                        "Erfinde keine Shop-Daten. Frage Daten zuerst über Tools ab. "
                        "Antworte auf Deutsch."
                    ),
                },
                {
                    "role": "user",
                    "content": question,
                },
            ]

            for _ in range(max_rounds):
                response = client.responses.create(
                    model=AI_MODEL,
                    input=input_items,
                    tools=openai_tools,
                )

                function_calls = [
                    item for item in response.output
                    if item.type == "function_call"
                ]

                if not function_calls:
                    return response.output_text

                input_items += response.output

                for call in function_calls:
                    args = json.loads(call.arguments or "{}")

                    tool_result = await session.call_tool(
                        call.name,
                        arguments=args,
                    )

                    input_items.append({
                        "type": "function_call_output",
                        "call_id": call.call_id,
                        "output": mcp_result_to_text(tool_result),
                    })

            return "Abgebrochen: zu viele Tool-Runden."

In [ ]:
answer = await ask_mcp(
    "Teste catalog_list_items und gib die gefundenen Artikel kompakt aus."
)

print(answer)

Beispiele von Abfragen:

In [ ]:
answer = await ask_mcp(
    "Liste alle Catalog Items. Gib pro Item ID, Name, Preis und den Rohinhalt aus, falls Felder fehlen."
)

print(answer)

In [ ]:
answer = await ask_mcp(
    "Wieviel Wert haben alle Catalog Items zusammen."
)

print(answer)

In [ ]:
answer = await ask_mcp(
    "Prüfe alle Orders gegen Customers und Catalog. Melde Orders, deren Customer oder Artikel nicht gefunden werden."
)

print(answer)

In [ ]:
answer = await ask_mcp(
    "Wieviel Umsatz habe ich erzielt"
)

print(answer)

In [ ]:
answer = await ask_mcp(
    "Kannst Du den Umsatz pro Kunde und Produkt aufschlüsseln?"
)

print(answer)

### Aufräumen

In [ ]:
%%bash
kubectl delete --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/mcp-server-deployment.yaml 
kubectl delete --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/mcp-server-service.yaml
kubectl delete ns ms-mcp